# Rolling Horizon Validierung

Dieses Notebook validiert die korrekte Ausführung des Rolling Horizon (RH) Algorithmus.

## Validierungs-Checks:
1. ✅ **Fenster-Struktur**: Anzahl und Überlappung der Optimierungsfenster
2. ✅ **SOC-Kontinuität**: Speicher-Zustand an Fenstergrenzen
3. ✅ **Kosten-Vergleich**: PF vs. RH (Optimality Gap)
4. ✅ **Design-Fixierung**: Überprüfung ob Design korrekt weitergegeben wird
5. ✅ **Überlappungs-Analyse**: Wie unterscheiden sich Entscheidungen in überlappenden Bereichen?

---

## 1. Setup

In [ ]:
# Auto-Setup: Projekt-Root finden
from pathlib import Path
import sys
import os

def find_project_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / '.git').exists() and (candidate / 'energis').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Projekt-Root: {PROJECT_ROOT}")

In [ ]:
# Imports
import warnings
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

from energis.run import rolling_horizon as rh
from energis.run.orchestrator import _collect_timeseries_and_summary

warnings.filterwarnings('ignore')

# Plot-Styling
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Imports erfolgreich")

## 2. Konfiguration

Konfiguriere hier deine Test-Szenarien mit verschiedenen RH-Parametern.

In [ ]:
# Basis-Konfiguration
BASE_CONFIGS = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/baseline.system.yaml',
    'configs/scenarios/pf_then_rh.workflow.scenario.yaml',
]

# RH-Parameter zum Testen
# Du kannst hier verschiedene Konfigurationen definieren
RH_SCENARIOS = [
    {
        'name': 'RH_168h_24h',
        'description': '1 Woche Horizont, 1 Tag Commit',
        'overrides': {
            'scenario': {
                'rolling_horizon': {
                    'heat_horizon_hours': 168.0,
                    'step_hours': 24.0,
                    'terminal_policy': 'free'
                }
            }
        }
    },
    {
        'name': 'RH_72h_24h',
        'description': '3 Tage Horizont, 1 Tag Commit',
        'overrides': {
            'scenario': {
                'rolling_horizon': {
                    'heat_horizon_hours': 72.0,
                    'step_hours': 24.0,
                    'terminal_policy': 'free'
                }
            }
        }
    },
]

print(f"📋 {len(RH_SCENARIOS)} Szenarien konfiguriert:")
for scenario in RH_SCENARIOS:
    print(f"  • {scenario['name']}: {scenario['description']}")

## 3. Optimierung ausführen

**Hinweis:** Dies kann einige Minuten dauern, je nach Szenariogröße.

In [ ]:
%%time
results = {}

for scenario in RH_SCENARIOS:
    print(f"\n{'='*70}")
    print(f"🚀 {scenario['name']}: {scenario['description']}")
    print('='*70)
    
    try:
        workflow = rh.run_workflow(BASE_CONFIGS, overrides=scenario['overrides'])
        results[scenario['name']] = {
            'workflow': workflow,
            'success': True,
            'error': None
        }
        
        # Quick summary
        if workflow.rh_result:
            n_windows = len(workflow.rh_result.windows)
            rh_cost = workflow.rh_result.costs.get('objective.OBJ_value_EUR', 0)
            print(f"✅ Erfolgreich: {n_windows} Fenster, Kosten: {rh_cost:,.0f} EUR")
        
    except Exception as e:
        print(f"❌ Fehler: {e}")
        results[scenario['name']] = {
            'workflow': None,
            'success': False,
            'error': str(e)
        }

print(f"\n{'='*70}")
print(f"✅ {sum(1 for r in results.values() if r['success'])}/{len(results)} Szenarien erfolgreich")
print('='*70)

## 4. Validierung: Fenster-Struktur

Überprüfung der Anzahl und Struktur der Optimierungsfenster.

In [ ]:
print("\n📊 FENSTER-STRUKTUR ANALYSE\n" + "="*70)

for scenario_name, result_data in results.items():
    if not result_data['success']:
        print(f"\n❌ {scenario_name}: Fehlgeschlagen")
        continue
    
    workflow = result_data['workflow']
    if not workflow.rh_result:
        continue
    
    print(f"\n🔍 {scenario_name}")
    print("-" * 70)
    
    rh_result = workflow.rh_result
    windows = rh_result.windows
    
    print(f"Anzahl Fenster:        {len(windows)}")
    print(f"Gesamt-Zeitschritte:   {len(rh_result.table)}")
    
    # Fenster-Details
    print(f"\nErste 5 Fenster:")
    for i, window in enumerate(windows[:5]):
        print(f"  Fenster {i:2d}: Start={window.start_index:4d}, "
              f"Commit={window.commit_steps:3d} Steps, "
              f"Kosten={window.costs.get('objective.OBJ_value_EUR', 0):,.0f} EUR")
    
    if len(windows) > 5:
        print(f"  ... [{len(windows) - 5} weitere Fenster]")
    
    # Überprüfung: Sind alle Zeitschritte abgedeckt?
    total_committed = sum(w.commit_steps for w in windows)
    expected = len(rh_result.table)
    
    if total_committed == expected:
        print(f"\n✅ Alle Zeitschritte korrekt committed ({total_committed} = {expected})")
    else:
        print(f"\n⚠️  Committed Mismatch: {total_committed} vs. {expected} erwartet")

print("\n" + "="*70)

## 5. Validierung: SOC-Kontinuität

**Kritischer Test:** Der Speicher-SOC muss an Fenstergrenzen kontinuierlich sein.

In [ ]:
print("\n🔋 SOC-KONTINUITÄTS-ANALYSE\n" + "="*70)

for scenario_name, result_data in results.items():
    if not result_data['success']:
        continue
    
    workflow = result_data['workflow']
    if not workflow.rh_result:
        continue
    
    rh_result = workflow.rh_result
    soc = rh_result.series.get('TES_SOC_MWh')
    
    if not soc:
        print(f"\n⚠️  {scenario_name}: Kein SOC verfügbar")
        continue
    
    print(f"\n📈 {scenario_name}")
    print("-" * 70)
    
    # Berechne Fenstergrenzen
    window_boundaries = []
    cumulative_steps = 0
    for window in rh_result.windows:
        cumulative_steps += window.commit_steps
        window_boundaries.append(cumulative_steps)
    
    # Überprüfe Sprünge an Fenstergrenzen
    max_jump = 0.0
    jumps_detected = []
    
    for boundary in window_boundaries[:-1]:  # Letzter Punkt = Ende, nicht prüfen
        if boundary > 0 and boundary < len(soc):
            jump = abs(soc[boundary] - soc[boundary - 1])
            max_jump = max(max_jump, jump)
            if jump > 0.01:  # Toleranz: 10 kWh
                jumps_detected.append((boundary, jump))
    
    # Ergebnis
    if not jumps_detected:
        print(f"✅ SOC kontinuierlich (max. Sprung: {max_jump:.4f} MWh)")
    else:
        print(f"⚠️  {len(jumps_detected)} Sprünge detektiert:")
        for idx, jump in jumps_detected[:5]:
            print(f"    Zeitschritt {idx}: Sprung = {jump:.3f} MWh")
    
    # Visualisierung
    fig, ax = plt.subplots(figsize=(14, 5))
    
    # SOC plotten
    ax.plot(soc, linewidth=1.5, label='SOC', color='#2E86AB')
    
    # Fenstergrenzen markieren
    for boundary in window_boundaries[:-1]:
        ax.axvline(boundary, color='red', alpha=0.4, linestyle='--', linewidth=1)
    
    ax.set_xlabel('Zeitschritt')
    ax.set_ylabel('SOC [MWh]')
    ax.set_title(f'Speicher-SOC: {scenario_name}\n(Rote Linien = Fenstergrenzen)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("\n" + "="*70)

## 6. Validierung: Optimality Gap (PF vs. RH)

**Erwartet:** RH-Kosten ≥ PF-Kosten (da RH weniger Information hat)

In [ ]:
print("\n💰 OPTIMALITY GAP ANALYSE\n" + "="*70)

gap_data = []

for scenario_name, result_data in results.items():
    if not result_data['success']:
        continue
    
    workflow = result_data['workflow']
    
    if not workflow.pf_result or not workflow.rh_result:
        print(f"⚠️  {scenario_name}: PF oder RH fehlt")
        continue
    
    pf_cost = workflow.pf_result.costs.get('objective.OBJ_value_EUR', 0)
    rh_cost = workflow.rh_result.costs.get('objective.OBJ_value_EUR', 0)
    
    if pf_cost > 0:
        gap_percent = ((rh_cost - pf_cost) / pf_cost) * 100
    else:
        gap_percent = 0.0
    
    gap_data.append({
        'scenario': scenario_name,
        'pf_cost': pf_cost,
        'rh_cost': rh_cost,
        'gap_percent': gap_percent
    })
    
    print(f"\n📊 {scenario_name}")
    print("-" * 70)
    print(f"PF Kosten:  {pf_cost:>15,.2f} EUR  (optimales Benchmark)")
    print(f"RH Kosten:  {rh_cost:>15,.2f} EUR  (myopische Planung)")
    print(f"Differenz:  {rh_cost - pf_cost:>15,.2f} EUR")
    print(f"Gap:        {gap_percent:>15,.2f} %")
    
    # Interpretation
    if gap_percent < 0:
        print("\n⚠️  WARNUNG: RH ist besser als PF (sollte nicht vorkommen!)")
    elif gap_percent < 1:
        print("\n✅ Exzellent: Gap < 1%")
    elif gap_percent < 5:
        print("\n✅ Gut: Gap < 5%")
    elif gap_percent < 10:
        print("\n⚠️  Akzeptabel: Gap < 10%")
    else:
        print("\n⚠️  Hoch: Gap > 10% - Horizont eventuell zu kurz")

# Vergleichs-Plot
if gap_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    scenarios = [d['scenario'] for d in gap_data]
    pf_costs = [d['pf_cost'] for d in gap_data]
    rh_costs = [d['rh_cost'] for d in gap_data]
    gaps = [d['gap_percent'] for d in gap_data]
    
    x = np.arange(len(scenarios))
    width = 0.35
    
    # Kosten-Vergleich
    ax1.bar(x - width/2, pf_costs, width, label='PF', color='#2E86AB')
    ax1.bar(x + width/2, rh_costs, width, label='RH', color='#A23B72')
    ax1.set_ylabel('Kosten [EUR]')
    ax1.set_title('Kosten-Vergleich: PF vs. RH')
    ax1.set_xticks(x)
    ax1.set_xticklabels(scenarios, rotation=45, ha='right')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Optimality Gap
    bars = ax2.bar(x, gaps, color='#F18F01')
    ax2.set_ylabel('Optimality Gap [%]')
    ax2.set_title('Optimality Gap (höher = mehr Unsicherheit)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(scenarios, rotation=45, ha='right')
    ax2.axhline(y=5, color='red', linestyle='--', alpha=0.5, label='5% Schwelle')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

print("\n" + "="*70)

## 7. Validierung: Design-Fixierung

Überprüft, ob das Design aus PF korrekt in RH übernommen wird.

In [ ]:
print("\n🏭 DESIGN-FIXIERUNGS-ANALYSE\n" + "="*70)

for scenario_name, result_data in results.items():
    if not result_data['success']:
        continue
    
    workflow = result_data['workflow']
    
    if not workflow.pf_result or not workflow.rh_result:
        continue
    
    print(f"\n🔍 {scenario_name}")
    print("-" * 70)
    
    # KORREKTUR: Design wird auf Workflow-Ebene gespeichert, nicht in pf_result
    # workflow.design wird vom PF-Step gesetzt
    # workflow.rh_result.design wird aus der RH-Optimierung extrahiert
    pf_design = workflow.design
    rh_design = workflow.rh_result.design if workflow.rh_result else None
    
    if not pf_design:
        print("⚠️  Kein Design verfügbar (workflow.design ist None)")
        print("    Hinweis: Design wird nur bei PF oder Design-Optimierung erstellt")
        continue
    
    # Wärmepumpen
    if pf_design.heat_pumps:
        print("\nWärmepumpen:")
        for hp_id, pf_data in sorted(pf_design.heat_pumps.items()):
            pf_capacity = pf_data.get('capacity_mw', 0)
            
            if rh_design and rh_design.heat_pumps:
                rh_data = rh_design.heat_pumps.get(hp_id, {})
                rh_capacity = rh_data.get('capacity_mw', 0)
                
                if abs(pf_capacity - rh_capacity) < 0.001:
                    print(f"  ✅ {hp_id}: {pf_capacity:.2f} MW (identisch)")
                else:
                    print(f"  ⚠️  {hp_id}: PF={pf_capacity:.2f} MW, RH={rh_capacity:.2f} MW (Abweichung!)")
            else:
                print(f"  ℹ️  {hp_id}: {pf_capacity:.2f} MW (RH-Design nicht extrahiert)")
    
    # Speicher
    if pf_design.storage:
        pf_storage_capacity = pf_design.storage.get('capacity_mwh', 0)
        print(f"\nSpeicher:")
        
        if rh_design and rh_design.storage:
            rh_storage_capacity = rh_design.storage.get('capacity_mwh', 0)
            
            if abs(pf_storage_capacity - rh_storage_capacity) < 0.001:
                print(f"  ✅ {pf_storage_capacity:.2f} MWh (identisch)")
            else:
                print(f"  ⚠️  PF={pf_storage_capacity:.2f} MWh, RH={rh_storage_capacity:.2f} MWh (Abweichung!)")
        else:
            print(f"  ℹ️  {pf_storage_capacity:.2f} MWh (RH-Design nicht extrahiert)")

print("\n" + "="*70)

## 8. Erweiterte Analyse: Überlappungs-Effekt

Untersucht, wie sich Entscheidungen in überlappenden Bereichen unterscheiden.

In [ ]:
print("\n🔄 ÜBERLAPPUNGS-ANALYSE\n" + "="*70)
print("\nVergleicht die erste Entscheidung in Fenster N+1 mit der Look-Ahead")
print("Entscheidung aus Fenster N (falls Überlappung existiert).\n")

for scenario_name, result_data in results.items():
    if not result_data['success']:
        continue
    
    workflow = result_data['workflow']
    
    if not workflow.rh_result:
        continue
    
    windows = workflow.rh_result.windows
    
    if len(windows) < 2:
        print(f"⚠️  {scenario_name}: Nur {len(windows)} Fenster (mindestens 2 benötigt)")
        continue
    
    print(f"\n📊 {scenario_name}")
    print("-" * 70)
    
    # Vergleiche erstes committed Fenster mit zweitem
    w0 = windows[0]
    w1 = windows[1]
    
    # Überlappende Region: Die Zeitschritte in w1 die bereits in w0 enthalten waren
    # aber nicht committed wurden
    overlap_start = w1.start_index
    w0_end = w0.start_index + len(w0.table)
    
    if overlap_start < w0_end:
        overlap_size = w0_end - overlap_start
        print(f"Fenster 0: Start={w0.start_index}, Länge={len(w0.table)}, Commit={w0.commit_steps}")
        print(f"Fenster 1: Start={w1.start_index}, Länge={len(w1.table)}")
        print(f"Überlappung: {overlap_size} Zeitschritte\n")
        
        # Beispiel: Vergleiche SOC in überlappender Region
        if 'TES_SOC_MWh' in w0.series and 'TES_SOC_MWh' in w1.series:
            w0_soc = w0.series['TES_SOC_MWh']
            w1_soc = w1.series['TES_SOC_MWh']
            
            # Index in den jeweiligen Fenstern
            idx_in_w0 = overlap_start - w0.start_index
            idx_in_w1 = 0  # Anfang von w1
            
            if idx_in_w0 < len(w0_soc) and idx_in_w1 < len(w1_soc):
                soc_w0 = w0_soc[idx_in_w0]
                soc_w1 = w1_soc[idx_in_w1]
                diff = abs(soc_w0 - soc_w1)
                
                print(f"SOC am Überlappungs-Start (global idx={overlap_start}):")
                print(f"  Fenster 0 (Look-ahead): {soc_w0:.3f} MWh")
                print(f"  Fenster 1 (Re-optimiert): {soc_w1:.3f} MWh")
                print(f"  Differenz: {diff:.3f} MWh")
                
                if diff < 0.01:
                    print("  ✅ Sehr ähnlich (konsistente Planung)")
                elif diff < 0.5:
                    print("  ⚠️  Geringe Abweichung (normale Re-Optimierung)")
                else:
                    print("  ⚠️  Größere Abweichung (starke Re-Optimierung)")
    else:
        print(f"Keine Überlappung zwischen Fenster 0 und 1 erkannt.")

print("\n" + "="*70)

## 9. Zusammenfassung

Gesamtbewertung aller durchgeführten Tests.

In [ ]:
print("\n" + "="*70)
print("📋 VALIDIERUNGS-ZUSAMMENFASSUNG")
print("="*70)

successful = sum(1 for r in results.values() if r['success'])
failed = len(results) - successful

print(f"\n✅ Erfolgreich: {successful}/{len(results)} Szenarien")
if failed > 0:
    print(f"❌ Fehlgeschlagen: {failed}/{len(results)} Szenarien")

print("\n🔍 Durchgeführte Checks:")
print("  1. ✅ Fenster-Struktur validiert")
print("  2. ✅ SOC-Kontinuität überprüft")
print("  3. ✅ Optimality Gap berechnet")
print("  4. ✅ Design-Fixierung validiert")
print("  5. ✅ Überlappungs-Effekte analysiert")

print("\n💡 Empfehlungen:")
print("  • Bei hohem Gap (>10%): Horizont verlängern oder step_hours reduzieren")
print("  • Bei SOC-Sprüngen: Terminal Policy überprüfen")
print("  • Für Produktionsläufe: PF_THEN_RH mit fix_design=true verwenden")

print("\n" + "="*70)
print("✅ Validierung abgeschlossen")
print("="*70)

---

## Weitere Analysen

Für detailliertere Visualisierungen siehe:
- `runner.ipynb` - Vollständiger Export mit allen Plots
- `scenario_studio.ipynb` - Parameter-Sweeps und Vergleiche

## Export der Ergebnisse

Die vollständigen Ergebnisse können mit folgendem Code exportiert werden:

In [ ]:
# Optional: Vollständiger Export
# from energis.run import orchestrator
# 
# for scenario in RH_SCENARIOS:
#     export_meta = orchestrator.run_all(BASE_CONFIGS, overrides=scenario['overrides'])
#     print(f"Exported to: {export_meta['outdir']}")